# Module 3 — Code Along: OOP, Dataclasses & FastAPI (the bank-accounts story)

Module 2 left an account as a `dict` + loose `insert/update/fetch` functions. Today we bundle the data **and** its behavior into a **class** — build it by hand, shortcut it with `@dataclass`, specialize with **inheritance**, and expose it over HTTP with **FastAPI**.

Section by section, one code cell per beat — run top to bottom:

**Section 1** the class by hand · **Section 2** @dataclass · **Section 3** inheritance · **Section 4** FastAPI

## Section 1 — OOP: the class by hand

A `dict` holds data but no behavior and no guardrails. A class bundles the data with the operations that protect it: `class` + `__init__` + `self`, attributes, and methods (one of which `raise`s to enforce a rule).

In [ ]:
# 1.1 · Why a class — a dict holds data but no behavior and no guardrails.
acct = {"owner": "Ada", "balance": 1500.0}
acct["balance"] = -50      # nothing stops an invalid balance
acct["ownr"] = "typo"      # a typo'd key is silently accepted
print(acct)                # no protection at all

In [ ]:
# 1.2 · class + __init__ + self — __init__ builds each object's data; self IS that object.
class BankAccount:
    def __init__(self, id, owner, balance):
        self.id = id
        self.owner = owner
        self.balance = balance

ada = BankAccount(1, "Ada", 1500.0)   # __init__ runs, returns the object
print(ada.owner, ada.balance)         # Ada 1500.0
print(ada)                            # <...BankAccount object at 0x...>  (ugly repr — fixed in Section 2)

In [ ]:
# 1.3 · Attributes — per-object data, read/written with a dot (a typo is an error, not a silent None).
print(ada.balance)        # 1500.0   (vs a dict's ada["balance"])
ada.owner = "Ada K."      # set with the dot
print(ada.owner)          # Ada K.
try:
    print(ada.ownr)       # typo'd attribute
except AttributeError as err:
    print("caught:", err) # caught: 'BankAccount' object has no attribute 'ownr'

In [ ]:
# 1.4 · Methods — functions inside the class (first param self); a method can raise to enforce a rule.
class BankAccount:
    def __init__(self, id, owner, balance):
        self.id = id; self.owner = owner; self.balance = balance
    def deposit(self, amount):
        self.balance += amount
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")   # the rule lives with the data
        self.balance -= amount

ada = BankAccount(1, "Ada", 1500.0)
ada.deposit(500);  print(ada.balance)    # 2000.0
ada.withdraw(300); print(ada.balance)    # 1700.0
try:
    ada.withdraw(9999)
except ValueError as err:
    print("declined:", err)              # declined: insufficient funds

## Section 2 — @dataclass: the clean record

Writing `__init__`/`__repr__`/`__eq__` by hand for every field is boilerplate. `@dataclass` writes them for you — and it's the next *data structure* after the dict: a typed, fixed-shape record.

In [ ]:
# 2.1 · The boilerplate — a hand-written __init__ for six fields is tedious and error-prone.
class BankAccount:
    def __init__(self, id, owner, account_type, balance, is_active, tags):
        self.id = id
        self.owner = owner
        self.account_type = account_type
        self.balance = balance
        self.is_active = is_active
        self.tags = tags          # ...and you'd still hand-write __repr__ and __eq__ too

a = BankAccount(1, "Ada", "savings", 1500.0, True, ["primary"])
print(a.owner)   # Ada — works, but that was a lot of typing

In [ ]:
# 2.2 · @dataclass — list the typed fields; get __init__, a readable __repr__, and __eq__ for free.
from dataclasses import dataclass, field

@dataclass
class BankAccount:
    id: int
    owner: str
    balance: float

ada = BankAccount(1, "Ada", 1500.0)
print(ada)                                    # BankAccount(id=1, owner='Ada', balance=1500.0)  <- __repr__
print(ada == BankAccount(1, "Ada", 1500.0))   # True  <- __eq__ by value
# the @ is a decorator — how it works is its own module; here it's a shortcut.

In [ ]:
# 2.3 · The next data structure: variable (M1) -> list & dict (M2) -> dataclass (M3).
# Each field is named AND typed — the fix for M2's mixed-value dict[str, object].
@dataclass
class BankAccount:
    id: int
    owner: str
    account_type: str
    balance: float

print(BankAccount(1, "Ada", "savings", 1500.0))

In [ ]:
# 2.4 · Defaults come after the required fields. A MUTABLE default (a list) needs default_factory.
@dataclass
class BankAccount:
    id: int
    owner: str
    balance: float
    is_active: bool = True
    tags: list[str] = field(default_factory=list)   # NEVER tags: list = []  (shared across instances)

a = BankAccount(2, "Lin", 800.0)
b = BankAccount(3, "Sam", 40.0)
a.tags.append("vip")
print(a.tags, b.tags)   # ['vip'] []  — separate lists, the M1 trap avoided

In [ ]:
# 2.5 · A dataclass is still a class — methods sit alongside the fields (data + behavior together).
@dataclass
class BankAccount:
    id: int
    owner: str
    balance: float = 0.0
    is_active: bool = True
    tags: list[str] = field(default_factory=list)
    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount

ada = BankAccount(1, "Ada", 1500.0)
ada.withdraw(200)
print(ada)              # BankAccount(id=1, owner='Ada', balance=1300.0, is_active=True, tags=[])

## Section 3 — Inheritance

A `SavingsAccount` **is-a** `BankAccount` plus extra. The subclass inherits the parent's fields and methods, and can `override` a method — calling `super()` to reuse the parent's logic.

In [ ]:
# 3.1 · Inheritance — a SavingsAccount IS-A BankAccount plus a rate.
@dataclass
class SavingsAccount(BankAccount):
    rate: float = 0.04        # inherits all of BankAccount's fields, adds one

s = SavingsAccount(1, "Ada", 1500.0)
print(s)                            # inherited fields + rate=0.04
print(isinstance(s, BankAccount))   # True — it IS-A BankAccount

In [ ]:
# 3.2 · A subclass gets the parent's fields AND methods for free.
s = SavingsAccount(2, "Lin", 1000.0)
s.withdraw(100)                # BankAccount.withdraw, inherited as-is
print(s.balance)               # 900.0

In [ ]:
# 3.3 · Override to specialize; super() reuses the parent's logic, then you add to it.
@dataclass
class SavingsAccount(BankAccount):
    rate: float = 0.04
    def add_interest(self):
        self.balance += self.balance * self.rate
    def withdraw(self, amount):
        super().withdraw(amount)   # reuse the parent's guardrail
        self.balance -= 1          # then apply a savings withdrawal fee

s = SavingsAccount(1, "Ada", 1500.0)
s.add_interest(); print(round(s.balance, 2))   # 1560.0  (1500 * 1.04)
s.withdraw(60);   print(s.balance)             # 1499.0  (1560 - 60 - 1)

In [ ]:
# 3.4 · Same is-a, in a library: Day 2's models inherit from Pydantic's BaseModel (validation for free).
try:
    from pydantic import BaseModel
    class Product(BaseModel):          # is-a BaseModel
        id: int
        name: str
    print(Product(id=1, name="USB-C Cable"))   # validated on construction
    try:
        Product(id="not-an-int", name="x")     # bad type -> raises
    except Exception as e:
        print("validation caught:", type(e).__name__)   # ValidationError
except ImportError:
    print("(pydantic arrives on Day 2 — same 'class inherits BaseModel' idea, plus validation)")

## Section 4 — FastAPI: expose the catalog

A route is just a function with `@app.get(...)` on top. We build GET/POST routes over the accounts and exercise them in-process with `TestClient` (in production you'd run `uv run uvicorn ... --reload` and open `/docs`).

In [ ]:
# 4.1 · A route is a function with @app.get on top — it runs when a request hits that path.
from fastapi import FastAPI
app = FastAPI()

ACCOUNTS = {1: {"id": 1, "owner": "Ada", "balance": 1500.0},
            2: {"id": 2, "owner": "Lin", "balance": 800.0}}

@app.get("/accounts")
def list_accounts():
    return list(ACCOUNTS.values())
# the @ is a decorator again — you hand list_accounts to FastAPI and it becomes a route (you *use* decorators, never write them).
print("route registered: GET /accounts")

In [ ]:
# 4.2 · Build the server — a path parameter to fetch one; POST to create.
@app.get("/accounts/{id}")
def get_account(id: int):
    return ACCOUNTS[id]

@app.post("/accounts")
def create(acct: dict):
    ACCOUNTS[acct["id"]] = acct
    return acct
print("routes registered: GET /accounts/{id}, POST /accounts")

In [ ]:
# 4.3 · Run it. In production: `uv run uvicorn catalog.server:app --reload`, then open http://localhost:8000/docs
# Here we call the app in-process with TestClient (no live server needed).
from fastapi.testclient import TestClient
client = TestClient(app)

print(client.get("/accounts/1").json())          # {'id': 1, 'owner': 'Ada', 'balance': 1500.0}
print(client.post("/accounts", json={"id": 3, "owner": "Sam", "balance": 40.0}).json())
print(client.get("/accounts").json())            # now includes Sam — FastAPI serialised it to JSON

## Next — Day 2: Pydantic

Our `@dataclass` still accepts bad data silently (`balance=-50` passes). **Next: Pydantic's `BaseModel`** — same fields, but **validated on construction** — wired into this FastAPI server so bad JSON returns a clean `422`.

And the `@` decorators we used as given (`@dataclass`, `@app.get`) return on Day 3 (`@pytest.fixture`) and Day 4 (`@agent.tool`) — always the same idea: you *use* a decorator to hand your function to a framework; you never write your own.